In [1]:
#fill NA with None
from sklearn import tree
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import random
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
from statistics import mean
from sklearn.naive_bayes import GaussianNB

In [2]:
#https://stackoverflow.com/questions/20224526/how-to-extract-the-decision-rules-from-scikit-learn-decision-tree
def get_code(tree, feature_names):
        left      = tree.tree_.children_left
        right     = tree.tree_.children_right
        threshold = tree.tree_.threshold
        features  = [feature_names[i] for i in tree.tree_.feature]
        value = tree.tree_.value

        def recurse(left, right, threshold, features, node):
                if (threshold[node] != -2):
                        print("if ( " + features[node] + " <= " + str(threshold[node]) + " ) {")
                        if left[node] != -1:
                                recurse (left, right, threshold, features,left[node])
                        print( "} else {")
                        if right[node] != -1:
                                recurse (left, right, threshold, features,right[node])
                        print( "}")
                else:
                        print( "return " + str(value[node]))

        recurse(left, right, threshold, features, 0)

In [3]:
data = pd.read_excel('dataset_gen.xlsx') 
dataset_all_gen = pd.read_excel('dataset_all_gen.xlsx')
dataset_all_original = pd.read_excel('dataset_all_original.xlsx')
dataset_jaccard_gen = pd.read_excel('dataset_jaccard_gen.xlsx')
dataset_jaccard_original = pd.read_excel('dataset_jaccard_original.xlsx')
dataset_myroom_gen = pd.read_excel('dataset_myroom_gen.xlsx')
dataset_myroom_original = pd.read_excel('dataset_myroom_original.xlsx')
dataset_university_gen = pd.read_excel('dataset_university_gen.xlsx')
dataset_university_original = pd.read_excel('dataset_university_original.xlsx')

In [4]:
data = data.sample(frac = 1) 
dataset_all_gen = dataset_all_gen.sample(frac = 1) 
dataset_all_original = dataset_all_original.sample(frac = 1) 
dataset_jaccard_gen = dataset_jaccard_gen.sample(frac = 1) 
dataset_jaccard_original = dataset_jaccard_original.sample(frac = 1) 
dataset_myroom_gen = dataset_myroom_gen.sample(frac = 1) 
dataset_myroom_original = dataset_myroom_original.sample(frac = 1) 
dataset_university_gen = dataset_university_gen.sample(frac = 1) 
dataset_university_original = dataset_university_original.sample(frac = 1) 

In [5]:

data = data.reset_index(drop=True)
dataset_all_gen = dataset_all_gen.reset_index(drop=True)
dataset_all_original = dataset_all_original.reset_index(drop=True)
dataset_jaccard_gen = dataset_jaccard_gen.reset_index(drop=True)
dataset_jaccard_original = dataset_jaccard_original.reset_index(drop=True)
dataset_myroom_gen = dataset_myroom_gen.reset_index(drop=True)
dataset_myroom_original = dataset_myroom_original.reset_index(drop=True)
dataset_university_gen = dataset_university_gen.reset_index(drop=True)
dataset_university_original = dataset_university_original.reset_index(drop=True)

In [6]:
data.isnull().sum()

type             0
Name             0
Vendor           0
DNS           2304
mDNS          2704
NUPNP        11000
open_port     1731
SSDP_Type     2304
SSDP_t       11000
SSDP_v       11000
dtype: int64

In [7]:
data = data.fillna("NaN")

In [8]:
#fill Na
dataset_all_gen = dataset_all_gen.fillna("NaN")
dataset_all_original = dataset_all_original.fillna("NaN")
dataset_jaccard_gen = dataset_jaccard_gen.fillna("NaN")
dataset_jaccard_original = dataset_jaccard_original.fillna("NaN")
dataset_myroom_gen = dataset_myroom_gen.fillna("NaN")
dataset_myroom_original = dataset_myroom_original.fillna("NaN")
dataset_university_gen = dataset_university_gen.fillna("NaN")
dataset_university_original = dataset_university_original.fillna("NaN")

In [9]:
data.isnull().sum()

type         0
Name         0
Vendor       0
DNS          0
mDNS         0
NUPNP        0
open_port    0
SSDP_Type    0
SSDP_t       0
SSDP_v       0
dtype: int64

In [10]:
data.dtypes

type         object
Name         object
Vendor       object
DNS          object
mDNS         object
NUPNP        object
open_port    object
SSDP_Type    object
SSDP_t       object
SSDP_v       object
dtype: object

In [11]:
data = data.astype('category')

In [12]:
dataset_all_gen = dataset_all_gen.astype('category')
dataset_all_original = dataset_all_original.astype('category')
dataset_jaccard_gen = dataset_jaccard_gen.astype('category')
dataset_jaccard_original = dataset_jaccard_original.astype('category')
dataset_myroom_gen = dataset_myroom_gen.astype('category')
dataset_myroom_original = dataset_myroom_original.astype('category')
dataset_university_gen = dataset_university_gen.astype('category')
dataset_university_original = dataset_university_original.astype('category')

In [13]:
data.dtypes

type         category
Name         category
Vendor       category
DNS          category
mDNS         category
NUPNP        category
open_port    category
SSDP_Type    category
SSDP_t       category
SSDP_v       category
dtype: object

In [14]:
data

,type,Name,Vendor,DNS,mDNS,NUPNP,open_port,SSDP_Type,SSDP_t,SSDP_v
0,Non-IOT,Macbook3,"Apple, Inc.",vkontakte.ru,NaN,NaN,tcp/1883,M-SEARCH * HTTP/1.1,NaN,NaN
1,Cameras,Sonos,Sonos Inc,NaN,_voyatouch._tcp.local.,NaN,tcp/80,NaN,NaN,NaN
2,Appliances,iRobot,iRobot Corporation,dion.ne.jp,_bigtax._tcp.local.,NaN,tcp/443,NOTIFY * HTTP/1.1,NaN,NaN
3,Non-IOT,Samsung Galaxy Tab,SAMSUNG,naver.com,NaN,NaN,NaN,M-SEARCH * HTTP/1.1,NaN,NaN
4,Health-Monitor,Blipcare Blood Pressure meter,Blipcare,buzzfeed.com,_greenlam._tcp.local.,NaN,tcp/80,NOTIFY * HTTP/1.1,NaN,NaN
5,Non-IOT,Samsung s7,SAMSUNG,umich.edu,_zamit._udp.local.,NaN,tcp/22,M-SEARCH * HTTP/1.1,NaN,NaN
6,Health-Monitor,NEST Protect smoke alarm,Nest Labs Inc.,cnet.com,_job._tcp.local.,NaN,tcp/80,NOTIFY * HTTP/1.1,NaN,NaN
7,Hubs,Apple HomePod,"Apple, Inc.",themeforest.net,_vagram._udp.local.,NaN,tcp/1883,NOTIFY * HTTP/1.1,NaN,NaN
8,Non-IOT,Xiaomi smartphone,Xiaomi,google.ca,_holdlamis._tcp.local.,NaN,tcp/443,M-SEARCH * HTTP/1.1,NaN,NaN
9,Hubs,Philips HUE Hub,Philips Lighting BV,NaN,_zoolab._udp.local.,NaN,tcp/1883,NaN,NaN,NaN


In [15]:
y = data.drop_duplicates(keep='first')
y = y.drop(['Name'], axis=1)
y = y.drop_duplicates(keep='first')
# y = y.reset_index()
# shuffle the DataFrame rows
# y = y.sample(frac = 1) 


In [16]:
len(y)

10610

In [17]:
#drop_duplicates
dataset_all_gen_y = dataset_all_gen.drop_duplicates(keep='first')
dataset_all_original_y = dataset_all_original.drop_duplicates(keep='first')
dataset_jaccard_gen_y = dataset_jaccard_gen.drop_duplicates(keep='first')
dataset_jaccard_original_y = dataset_jaccard_original.drop_duplicates(keep='first')
dataset_myroom_gen_y = dataset_myroom_gen.drop_duplicates(keep='first')
dataset_myroom_original_y = dataset_myroom_original.drop_duplicates(keep='first')
dataset_university_gen_y = dataset_university_gen.drop_duplicates(keep='first')
dataset_university_original_y = dataset_university_original.drop_duplicates(keep='first')

In [18]:
#drop name
dataset_all_gen_y = dataset_all_gen_y.drop(['Name'], axis=1)
dataset_all_original_y = dataset_all_original_y.drop(['Name'], axis=1)
dataset_jaccard_gen_y = dataset_jaccard_gen_y.drop(['Name'], axis=1)
dataset_jaccard_original_y = dataset_jaccard_original_y.drop(['Name'], axis=1)
dataset_myroom_gen_y = dataset_myroom_gen_y.drop(['Name'], axis=1)
dataset_myroom_original_y = dataset_myroom_original_y.drop(['Name'], axis=1)
dataset_university_gen_y = dataset_university_gen_y.drop(['Name'], axis=1)
dataset_university_original_y = dataset_university_original_y.drop(['Name'], axis=1)

In [19]:
#drop_duplicates
dataset_all_gen_y = dataset_all_gen_y.drop_duplicates(keep='first')
dataset_all_original_y = dataset_all_original_y.drop_duplicates(keep='first')
dataset_jaccard_gen_y = dataset_jaccard_gen_y.drop_duplicates(keep='first')
dataset_jaccard_original_y = dataset_jaccard_original_y.drop_duplicates(keep='first')
dataset_myroom_gen_y = dataset_myroom_gen_y.drop_duplicates(keep='first')
dataset_myroom_original_y = dataset_myroom_original_y.drop_duplicates(keep='first')
dataset_university_gen_y = dataset_university_gen_y.drop_duplicates(keep='first')
dataset_university_original_y = dataset_university_original_y.drop_duplicates(keep='first')

In [20]:
y = y.apply(LabelEncoder().fit_transform)

In [21]:
dataset_all_gen_y = dataset_all_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_all_original_y = dataset_all_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_jaccard_gen_y = dataset_jaccard_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_jaccard_original_y = dataset_jaccard_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_myroom_gen_y = dataset_myroom_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_myroom_original_y = dataset_myroom_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_university_gen_y = dataset_university_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_university_original_y = dataset_university_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')

In [22]:
list_data = []
list_data.append(y)
list_data.append(dataset_all_gen_y)
list_data.append(dataset_all_original_y)
list_data.append(dataset_jaccard_gen_y)
list_data.append(dataset_jaccard_original_y)
list_data.append(dataset_myroom_gen_y)
list_data.append(dataset_myroom_original_y)
list_data.append(dataset_university_gen_y)
list_data.append(dataset_university_original_y)
len(list_data)

9

In [23]:
accuracy = []
accuracy_t = []
accuracy_temp = []

In [24]:
for data in list_data:
    for i in range(100):
        kf = KFold(n_splits=10, random_state=None, shuffle=True)
#         kf.get_n_splits(data)
        for train_index, test_index in kf.split(data):
        #     print(index)
#             print("TRAIN:", train_index, "TEST:", test_index)
            train, test = data.iloc[train_index], data.iloc[test_index]
#             print("TRAIN:", train, "TEST:", test)
        #     print(len(train))
#             print("TRAIN:", len(train), "TEST:", len(test))
            label_data = train.type
            train_data = train.drop(['type'], axis=1)
            tlabel_data = test.type
            test_data = test.drop(['type'], axis=1)
            truelabel_data = pd.DataFrame(tlabel_data)
            truelabel_data = truelabel_data.reset_index(drop=True)
            gnb = GaussianNB()
#             clf = tree.DecisionTreeClassifier()
#             clf = clf.fit(train_data,label_data )
            gnb = gnb.fit(train_data,label_data )
#             predict = clf.predict(test_data)
            predict = gnb.predict(test_data)
#             print(truelabel_data)
#             print(predict)
        #     len(predict)
            result_label_data = pd.DataFrame(columns = ['type'])
            result_label_data.type = predict
            compare = pd.DataFrame(columns = ['truelabel_data','predictedlabel'])
            compare.truelabel_data = truelabel_data.type
            compare.predictedlabel = result_label_data.type
#             print(compare)
            t = (accuracy_score(compare.truelabel_data, compare.predictedlabel))
            accuracy_temp.append(t)
            print(i,"  Accuracy: ",t)
        accuracy_t.append(mean(accuracy_temp))
    accuracy.append(mean(accuracy_t))

0   Accuracy:  0.5032987747408105
0   Accuracy:  0.49858623939679547
0   Accuracy:  0.472196041470311
0   Accuracy:  0.48539114043355325
0   Accuracy:  0.49293119698397736
0   Accuracy:  0.48539114043355325
0   Accuracy:  0.4872761545711593
0   Accuracy:  0.49104618284637136
0   Accuracy:  0.4957587181903864
0   Accuracy:  0.48633364750235625
1   Accuracy:  0.5080113100848256
1   Accuracy:  0.4872761545711593
1   Accuracy:  0.4882186616399623
1   Accuracy:  0.49104618284637136
1   Accuracy:  0.4882186616399623
1   Accuracy:  0.4637134778510839
1   Accuracy:  0.4806786050895382
1   Accuracy:  0.5023562676720076
1   Accuracy:  0.5004712535344015
1   Accuracy:  0.47879359095193214
2   Accuracy:  0.47879359095193214
2   Accuracy:  0.470311027332705
2   Accuracy:  0.4825636192271442
2   Accuracy:  0.473138548539114
2   Accuracy:  0.5089538171536286
2   Accuracy:  0.5268614514608859
2   Accuracy:  0.5014137606032045
2   Accuracy:  0.471253534401508
2   Accuracy:  0.48539114043355325
2   Accu

24   Accuracy:  0.5004712535344015
24   Accuracy:  0.4740810556079171
24   Accuracy:  0.49858623939679547
24   Accuracy:  0.4957587181903864
24   Accuracy:  0.47879359095193214
24   Accuracy:  0.46936852026390197
24   Accuracy:  0.4957587181903864
25   Accuracy:  0.5042412818096136
25   Accuracy:  0.5230914231856739
25   Accuracy:  0.5249764373232799
25   Accuracy:  0.470311027332705
25   Accuracy:  0.49952874646559847
25   Accuracy:  0.49670122525918947
25   Accuracy:  0.4580584354382658
25   Accuracy:  0.48350612629594725
25   Accuracy:  0.45523091423185674
25   Accuracy:  0.4759660697455231
26   Accuracy:  0.5089538171536286
26   Accuracy:  0.5032987747408105
26   Accuracy:  0.46936852026390197
26   Accuracy:  0.5193213949104618
26   Accuracy:  0.4957587181903864
26   Accuracy:  0.49198868991517436
26   Accuracy:  0.46277097078228085
26   Accuracy:  0.48633364750235625
26   Accuracy:  0.5023562676720076
26   Accuracy:  0.46936852026390197
27   Accuracy:  0.4872761545711593
27   Accu

47   Accuracy:  0.4891611687087653
47   Accuracy:  0.5146088595664468
47   Accuracy:  0.4759660697455231
47   Accuracy:  0.46182846371347785
48   Accuracy:  0.472196041470311
48   Accuracy:  0.49952874646559847
48   Accuracy:  0.48633364750235625
48   Accuracy:  0.49952874646559847
48   Accuracy:  0.5004712535344015
48   Accuracy:  0.49010367577756836
48   Accuracy:  0.4806786050895382
48   Accuracy:  0.49764373232799247
48   Accuracy:  0.47785108388312914
48   Accuracy:  0.5004712535344015
49   Accuracy:  0.47785108388312914
49   Accuracy:  0.47879359095193214
49   Accuracy:  0.5230914231856739
49   Accuracy:  0.4825636192271442
49   Accuracy:  0.4891611687087653
49   Accuracy:  0.4759660697455231
49   Accuracy:  0.49764373232799247
49   Accuracy:  0.48539114043355325
49   Accuracy:  0.4948162111215834
49   Accuracy:  0.5032987747408105
50   Accuracy:  0.47973609802073514
50   Accuracy:  0.5183788878416589
50   Accuracy:  0.4938737040527804
50   Accuracy:  0.4665409990574929
50   Accu

71   Accuracy:  0.5127238454288408
71   Accuracy:  0.48539114043355325
71   Accuracy:  0.4816211121583412
72   Accuracy:  0.532516493873704
72   Accuracy:  0.4825636192271442
72   Accuracy:  0.47690857681432614
72   Accuracy:  0.5117813383600377
72   Accuracy:  0.4590009425070688
72   Accuracy:  0.46936852026390197
72   Accuracy:  0.48350612629594725
72   Accuracy:  0.4938737040527804
72   Accuracy:  0.4806786050895382
72   Accuracy:  0.5098963242224317
73   Accuracy:  0.5042412818096136
73   Accuracy:  0.5127238454288408
73   Accuracy:  0.4882186616399623
73   Accuracy:  0.5042412818096136
73   Accuracy:  0.4825636192271442
73   Accuracy:  0.4825636192271442
73   Accuracy:  0.46182846371347785
73   Accuracy:  0.5155513666352498
73   Accuracy:  0.46748350612629597
73   Accuracy:  0.4891611687087653
74   Accuracy:  0.5117813383600377
74   Accuracy:  0.5051837888784166
74   Accuracy:  0.4599434495758718
74   Accuracy:  0.47785108388312914
74   Accuracy:  0.4882186616399623
74   Accuracy:

95   Accuracy:  0.4957587181903864
95   Accuracy:  0.4816211121583412
95   Accuracy:  0.49198868991517436
95   Accuracy:  0.4957587181903864
95   Accuracy:  0.48539114043355325
96   Accuracy:  0.4524033930254477
96   Accuracy:  0.5070688030160226
96   Accuracy:  0.4882186616399623
96   Accuracy:  0.4750235626767201
96   Accuracy:  0.48633364750235625
96   Accuracy:  0.4948162111215834
96   Accuracy:  0.48539114043355325
96   Accuracy:  0.5164938737040528
96   Accuracy:  0.48633364750235625
96   Accuracy:  0.49858623939679547
97   Accuracy:  0.5080113100848256
97   Accuracy:  0.48633364750235625
97   Accuracy:  0.4891611687087653
97   Accuracy:  0.4882186616399623
97   Accuracy:  0.49764373232799247
97   Accuracy:  0.48539114043355325
97   Accuracy:  0.49293119698397736
97   Accuracy:  0.49010367577756836
97   Accuracy:  0.471253534401508
97   Accuracy:  0.49952874646559847
98   Accuracy:  0.5023562676720076
98   Accuracy:  0.49198868991517436
98   Accuracy:  0.48539114043355325
98   Ac

19   Accuracy:  0.2595217006200177
19   Accuracy:  0.23737821080602303
19   Accuracy:  0.2391496899911426
19   Accuracy:  0.2612931798051373
19   Accuracy:  0.2400354295837024
19   Accuracy:  0.24379432624113476
19   Accuracy:  0.2429078014184397
19   Accuracy:  0.28102836879432624
20   Accuracy:  0.23560673162090345
20   Accuracy:  0.25686448184233834
20   Accuracy:  0.2595217006200177
20   Accuracy:  0.23737821080602303
20   Accuracy:  0.2630646589902569
20   Accuracy:  0.26660761736049604
20   Accuracy:  0.25066430469441986
20   Accuracy:  0.25177304964539005
20   Accuracy:  0.24468085106382978
20   Accuracy:  0.25177304964539005
21   Accuracy:  0.24800708591674048
21   Accuracy:  0.2550930026572188
21   Accuracy:  0.27192205491585475
21   Accuracy:  0.22940655447298494
21   Accuracy:  0.2657218777679362
21   Accuracy:  0.2329495128432241
21   Accuracy:  0.271036315323295
21   Accuracy:  0.26152482269503546
21   Accuracy:  0.25443262411347517
21   Accuracy:  0.2296099290780142
22   

... (24 outputs left out)

3   Accuracy:  0.5517241379310345
3   Accuracy:  0.5714285714285714
3   Accuracy:  0.5
3   Accuracy:  0.4642857142857143
3   Accuracy:  0.5
3   Accuracy:  0.5
3   Accuracy:  0.4642857142857143
4   Accuracy:  0.5517241379310345
4   Accuracy:  0.4827586206896552
4   Accuracy:  0.6551724137931034
4   Accuracy:  0.3448275862068966
4   Accuracy:  0.5357142857142857
4   Accuracy:  0.6428571428571429
4   Accuracy:  0.5
4   Accuracy:  0.6071428571428571
4   Accuracy:  0.6428571428571429
4   Accuracy:  0.6071428571428571
5   Accuracy:  0.5862068965517241
5   Accuracy:  0.41379310344827586
5   Accuracy:  0.5172413793103449
5   Accuracy:  0.5517241379310345
5   Accuracy:  0.5714285714285714
5   Accuracy:  0.32142857142857145
5   Accuracy:  0.6428571428571429
5   Accuracy:  0.5714285714285714
5   Accuracy:  0.5357142857142857
5   Accuracy:  0.5714285714285714
6   Accuracy:  0.5862068965517241
6   Accuracy:  0.4827586206896552
6   Accuracy:  0.5172413793103449
6   Accuracy:  0.5517241379310345
6   

27   Accuracy:  0.6071428571428571
27   Accuracy:  0.5714285714285714
27   Accuracy:  0.5
28   Accuracy:  0.3448275862068966
28   Accuracy:  0.5862068965517241
28   Accuracy:  0.6551724137931034
28   Accuracy:  0.5172413793103449
28   Accuracy:  0.5357142857142857
28   Accuracy:  0.5
28   Accuracy:  0.5357142857142857
28   Accuracy:  0.6428571428571429
28   Accuracy:  0.6785714285714286
28   Accuracy:  0.42857142857142855
29   Accuracy:  0.5862068965517241
29   Accuracy:  0.6206896551724138
29   Accuracy:  0.41379310344827586
29   Accuracy:  0.5862068965517241
29   Accuracy:  0.5
29   Accuracy:  0.4642857142857143
29   Accuracy:  0.5714285714285714
29   Accuracy:  0.6428571428571429
29   Accuracy:  0.6428571428571429
29   Accuracy:  0.5
30   Accuracy:  0.41379310344827586
30   Accuracy:  0.4827586206896552
30   Accuracy:  0.5517241379310345
30   Accuracy:  0.7241379310344828
30   Accuracy:  0.5714285714285714
30   Accuracy:  0.6071428571428571
30   Accuracy:  0.5714285714285714
30   Ac

52   Accuracy:  0.4482758620689655
52   Accuracy:  0.6785714285714286
52   Accuracy:  0.6071428571428571
52   Accuracy:  0.6071428571428571
52   Accuracy:  0.6428571428571429
52   Accuracy:  0.5
52   Accuracy:  0.5357142857142857
53   Accuracy:  0.4827586206896552
53   Accuracy:  0.4827586206896552
53   Accuracy:  0.6551724137931034
53   Accuracy:  0.7931034482758621
53   Accuracy:  0.5714285714285714
53   Accuracy:  0.35714285714285715
53   Accuracy:  0.4642857142857143
53   Accuracy:  0.5
53   Accuracy:  0.5357142857142857
53   Accuracy:  0.32142857142857145
54   Accuracy:  0.5172413793103449
54   Accuracy:  0.5862068965517241
54   Accuracy:  0.6206896551724138
54   Accuracy:  0.6206896551724138
54   Accuracy:  0.5714285714285714
54   Accuracy:  0.5
54   Accuracy:  0.32142857142857145
54   Accuracy:  0.7142857142857143
54   Accuracy:  0.4642857142857143
54   Accuracy:  0.6428571428571429
55   Accuracy:  0.5862068965517241
55   Accuracy:  0.41379310344827586
55   Accuracy:  0.48275862

77   Accuracy:  0.42857142857142855
78   Accuracy:  0.5517241379310345
78   Accuracy:  0.5517241379310345
78   Accuracy:  0.5862068965517241
78   Accuracy:  0.4482758620689655
78   Accuracy:  0.6071428571428571
78   Accuracy:  0.6785714285714286
78   Accuracy:  0.2857142857142857
78   Accuracy:  0.6785714285714286
78   Accuracy:  0.6428571428571429
78   Accuracy:  0.5
79   Accuracy:  0.4827586206896552
79   Accuracy:  0.5862068965517241
79   Accuracy:  0.6206896551724138
79   Accuracy:  0.5517241379310345
79   Accuracy:  0.4642857142857143
79   Accuracy:  0.35714285714285715
79   Accuracy:  0.5714285714285714
79   Accuracy:  0.6428571428571429
79   Accuracy:  0.5357142857142857
79   Accuracy:  0.6428571428571429
80   Accuracy:  0.6896551724137931
80   Accuracy:  0.5172413793103449
80   Accuracy:  0.6206896551724138
80   Accuracy:  0.5862068965517241
80   Accuracy:  0.42857142857142855
80   Accuracy:  0.6785714285714286
80   Accuracy:  0.39285714285714285
80   Accuracy:  0.4642857142857

In [25]:
print(len(y),"\t Accuracy for dataset_gen: \t", accuracy[0])
print(len(dataset_all_gen_y),"\t Accuracy for dataset_all_gen: \t", accuracy[1])
print(len(dataset_all_original_y),"\t Accuracy for dataset_all_original: \t", accuracy[2])
print(len(dataset_jaccard_gen_y),"\t Accuracy for dataset_jaccard_gen: \t", accuracy[3])
print(len(dataset_jaccard_original_y),"\t Accuracy for dataset_jaccard_original: \t", accuracy[4])
print(len(dataset_myroom_gen_y),"\t Accuracy for dataset_myroom_gen: \t", accuracy[5])
print(len(dataset_myroom_original_y),"\t Accuracy for dataset_myroom_original: \t", accuracy[6])
print(len(dataset_university_gen_y),"\t Accuracy for dataset_university_gen: \t", accuracy[7])
print(len(dataset_university_original_y),"\t Accuracy for dataset_university_original: \t", accuracy[8])

10610 	 Accuracy for dataset_gen: 	 0.49018223355653706
11287 	 Accuracy for dataset_all_gen: 	 0.45333489580944447
677 	 Accuracy for dataset_all_original: 	 0.4387006307522502
10912 	 Accuracy for dataset_jaccard_gen: 	 0.4326418752648154
302 	 Accuracy for dataset_jaccard_original: 	 0.43828081708242544
10701 	 Accuracy for dataset_myroom_gen: 	 0.44714768433325897
91 	 Accuracy for dataset_myroom_original: 	 0.45616621166462984
10894 	 Accuracy for dataset_university_gen: 	 0.4668834633715815
284 	 Accuracy for dataset_university_original: 	 0.4748799919668726
